## A1.1 User–Item Matrix Sparsity and Matrix Factorization

**Theoretical answer**

A user–item interaction matrix $R$ of size $2{,}500{,}000 \times 400{,}000$ is extremely sparse because each user interacts with only a tiny fraction of the full catalog. In an e-commerce platform, a typical user may browse, click, or purchase tens to hundreds of products, while the total catalog contains hundreds of thousands of SKUs. Therefore, most entries in $R$ are 0, meaning 'no interaction'.

Matrix factorization approximates the sparse matrix as:

$$R \approx U V^T$$

where:
- $U$ is the **user latent factor matrix**
- $V$ is the **item latent factor matrix**
- each user and item is represented in a smaller latent space

If the number of latent factors is $k = 50$:
- $U$ has shape **(2,500,000 × 50)**
- $V$ has shape **(400,000 × 50)**

This reduces dimensionality and helps predict missing interactions, which is useful in recommendation systems.

In [1]:
import numpy as np

# Example of user-item interaction matrix
R_demo = np.array([
    [1, 0, 0, 1, 0],
    [0, 0, 1, 0, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1],
])

total_entries = R_demo.size
non_zero_entries = np.count_nonzero(R_demo)
zero_entries = total_entries - non_zero_entries
sparsity = zero_entries / total_entries

print('Demo interaction matrix:\n', R_demo)
print(f'Total entries: {total_entries}')
print(f'Non-zero entries: {non_zero_entries}')
print(f'Zero entries: {zero_entries}')
print(f'Sparsity: {sparsity:.2%}')

Demo interaction matrix:
 [[1 0 0 1 0]
 [0 0 1 0 0]
 [1 1 0 0 0]
 [0 0 0 0 1]]
Total entries: 20
Non-zero entries: 6
Zero entries: 14
Sparsity: 70.00%


## A1.2 Cosine Similarity Between Two Users

**Problem**

User A = [1, 0, 1, 1, 0, 1, 0]  
User B = [1, 1, 0, 1, 0, 0, 1]

**Theoretical answer**

Cosine similarity is:

$$\cos(\theta) = \frac{A \cdot B}{\|A\|\|B\|}$$

Step-by-step:
- Dot product: $1\cdot1 + 0\cdot1 + 1\cdot0 + 1\cdot1 + 0\cdot0 + 1\cdot0 + 0\cdot1 = 2$
- $\|A\| = \sqrt{1^2+0^2+1^2+1^2+0^2+1^2+0^2} = \sqrt{4} = 2$
- $\|B\| = \sqrt{1^2+1^2+0^2+1^2+0^2+0^2+1^2} = \sqrt{4} = 2$

Therefore:

$$\cos(\theta) = \frac{2}{2\times2} = 0.5$$

A cosine similarity of **0.5** indicates moderate similarity. I would not blindly recommend exactly the same products to both users. Instead, I would treat them as partially similar and combine this signal with product content, recency, and browsing behavior.

In [2]:
import numpy as np

user_a = np.array([1, 0, 1, 1, 0, 1, 0])
user_b = np.array([1, 1, 0, 1, 0, 0, 1])

dot_product = np.dot(user_a, user_b)
norm_a = np.linalg.norm(user_a)
norm_b = np.linalg.norm(user_b)
cosine_similarity = dot_product / (norm_a * norm_b)

print('User A:', user_a)
print('User B:', user_b)
print('Dot product:', dot_product)
print('Norm of A:', norm_a)
print('Norm of B:', norm_b)
print('Cosine similarity:', round(cosine_similarity, 4))

User A: [1 0 1 1 0 1 0]
User B: [1 1 0 1 0 0 1]
Dot product: 2
Norm of A: 2.0
Norm of B: 2.0
Cosine similarity: 0.5


## A1.3 PCA and Explained Variance

**Problem**

Top 5 explained variance ratios are:
[0.38, 0.22, 0.14, 0.09, 0.07]

**Theoretical answer**

Cumulative explained variance:
- Component 1: 0.38
- Component 2: 0.60
- Component 3: 0.74
- Component 4: 0.83
- Component 5: 0.90

So the first 5 components explain **90%** of the variance, not 95%. Therefore, more than 5 components are needed to reach 95% variance. Based on the remaining variance, the final number would likely be around **6–8 components**, depending on the next ratios.

I would prefer PCA over manual feature selection when:
- there are many correlated features
- the raw feature space is high-dimensional
- I want a compact representation with less noise
- computational efficiency matters

For recommendation models, PCA is useful when many engineered features overlap or when dimensionality reduction improves speed without losing much information.

In [ ]:
import numpy as np

explained_variance_ratio = np.array([0.38, 0.22, 0.14, 0.09, 0.07])
cumulative_variance = np.cumsum(explained_variance_ratio)

for i, value in enumerate(cumulative_variance, start=1):
    print(f'Components {i}: cumulative explained variance = {value:.2f}')

reaches_95 = np.where(cumulative_variance >= 0.95)[0]
if len(reaches_95) > 0:
    print('Minimum components to reach 95% variance:', reaches_95[0] + 1)
else:
    print('Top 5 components only explain 90%, so more than 5 components are needed.')

Components 1: cumulative explained variance = 0.38
Components 2: cumulative explained variance = 0.60
Components 3: cumulative explained variance = 0.74
Components 4: cumulative explained variance = 0.83
Components 5: cumulative explained variance = 0.90
Top 5 components only explain 90%, so more than 5 components are needed.


### Estimating components needed for 95% variance

The top 5 components explain 90% cumulative variance:
[0.38, 0.22, 0.14, 0.09, 0.07] → cumsum = [0.38, 0.60, 0.74, 0.83, 0.90]

We need 5% more. To estimate how many additional components are required,
we observe that the variance ratios are decaying — each component explains
progressively less variance than the previous one.

Decay pattern from the given ratios:
  Component 2 / Component 1 = 0.22 / 0.38 ≈ 0.58
  Component 3 / Component 2 = 0.14 / 0.22 ≈ 0.64
  Component 4 / Component 3 = 0.09 / 0.14 ≈ 0.64
  Component 5 / Component 4 = 0.07 / 0.09 ≈ 0.78

The decay ratio stabilises around 0.64–0.78. Using 0.70 as a reasonable
estimate, we can extrapolate subsequent components:
  Component 6 ≈ 0.07 × 0.70 = 0.049
  Component 7 ≈ 0.049 × 0.70 = 0.034

Cumulative after component 6: 0.90 + 0.049 = 0.949 → still below 95%
Cumulative after component 7: 0.949 + 0.034 = 0.983 → exceeds 95%

Therefore approximately 7 components are needed to explain 95% variance.

Note: this is an estimate based on geometric decay. In practice you would
fit PCA on the actual data and read the exact number from the cumulative
explained_variance_ratio_ array directly.

In [2]:
import numpy as np

# Given variance ratios
given = np.array([0.38, 0.22, 0.14, 0.09, 0.07])
cumulative = np.cumsum(given)

print("Given components:")
for i, (v, c) in enumerate(zip(given, cumulative), 1):
    print(f"  Component {i}: variance={v:.2f}  cumulative={c:.2f}")

print(f"\nTop 5 components explain {cumulative[-1]:.0%} — need {1-cumulative[-1]:.0%} more\n")

# Estimate decay ratio from last 3 transitions (more stable than using all)
decay_ratios = given[1:] / given[:-1]
print("Decay ratios between consecutive components:")
for i, r in enumerate(decay_ratios, 2):
    print(f"  Component {i} / Component {i-1} = {r:.3f}")

decay = np.mean(decay_ratios[-3:])  # use last 3 for stability
print(f"\nEstimated decay factor (mean of last 3): {decay:.3f}")

# Extrapolate beyond component 5
components = list(given)
target = 0.95

comp = given[-1]
n = 5
while np.sum(components) < target:
    comp = comp * decay
    components.append(comp)
    n += 1
    print(f"  Component {n} ≈ {comp:.4f}  →  cumulative = {np.sum(components):.4f}")

print(f"\nEstimate: {n} components needed to explain 95% variance")
print(f"Final cumulative variance: {np.sum(components):.4f}")
print()
print("Note: in practice, read directly from fitted PCA:")
print("  pca = PCA().fit(X)")
print("  n = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1")

Given components:
  Component 1: variance=0.38  cumulative=0.38
  Component 2: variance=0.22  cumulative=0.60
  Component 3: variance=0.14  cumulative=0.74
  Component 4: variance=0.09  cumulative=0.83
  Component 5: variance=0.07  cumulative=0.90

Top 5 components explain 90% — need 10% more

Decay ratios between consecutive components:
  Component 2 / Component 1 = 0.579
  Component 3 / Component 2 = 0.636
  Component 4 / Component 3 = 0.643
  Component 5 / Component 4 = 0.778

Estimated decay factor (mean of last 3): 0.686
  Component 6 ≈ 0.0480  →  cumulative = 0.9480
  Component 7 ≈ 0.0329  →  cumulative = 0.9809

Estimate: 7 components needed to explain 95% variance
Final cumulative variance: 0.9809

Note: in practice, read directly from fitted PCA:
  pca = PCA().fit(X)
  n = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1
